# PyA GUI examples

- this notebook demonstrates GUI for interacting with pya and audio data
- the first GUI integrated is Scope, a realtime oscilloscope and freqscope for
  the Aserver.
- next an Aserver dashboard (control interface) and Asig viewer will be added

## Scope

- Scope offers a simple lightweight fast and unobtrusive realtime display 
for audio data. 
- the main purpose is to use it in Aserver for r/t-monitoring, yet Scope
  can also be created independent of it for any other display purpose
- to minimize interference with time critical components (such as Aserver)
  - Scope is started via subprocess as completely independent process
  - commands are sent via pipe and processed line by line
  - shared memory is used for data transmission, i.e. it involves merely a memcopy on the user's side - all subsequent computations and gui rendering happens in the separated GUI process.


In [ ]:
from pya.gui import Scope
scp = Scope(num_samples=512, num_channels=2, pos=(-400, 0), size=(400, 300), rate=20, mode="signal")

In [ ]:
# start the scope
scp.start()

In [ ]:
# update data as needed (this is also done in Aserver if scope is used)
import numpy as np 
scp.set_data(np.random.random((512, 2))-0.5)

In [ ]:
# move window programmatically
scp.move(-400, 100)

In [ ]:
# resize GUI window programmatically
scp.resize(320, 200)

In [ ]:
# change the GUI process render/update framerate
scp.render_framerate(50)

In [ ]:
# stop the thread (if needed)
scp.stop()

In [ ]:
# exit the GUI - this closes the process
scp.exit() # alternatively use del(scp)

## Scope as Oscilloscope in Aserver

Scope is integrated into Aserver via the scope_gui() method

In [ ]:
from pya import startup
from pya.agen.lib import WhiteNoise, Line, Env

In [ ]:
s = startup(bs=1024) 
s.scope_gui()

The scope is already running. Play anything via Asig.play() or using Agen...

In [ ]:
(WhiteNoise() * Env([0.5,0,0.5, 0], [1,1, 0.2])).dup(2).play();

The Scope instance can be accessed via s.scope. All functions demonstrated above
- move, resize, render_framerate, etc. - are available.

In [ ]:
s.scope.data # this is always the current data (i.e. shared memory)

In [ ]:
from pya import Asig

Asig(s.scope.data).plot(offset=0.2, lw=0.5)

- Note that Scope only updates the view when new data are set.
- In Aserver, once all events have been processed, no further data is written
- In consequence, the last written block remains visible in the GUI.
- Also note, that you can resize the window, zoom in, pan however you like!
  - The GUI uses pyqtgraph, clicking the right button gives access to various features
  - Try Plot Options -> Transforms -> PowerSpectrum for a spectrum
  - Try Plot Options -> 

In [ ]:
from pya import Asig
s.scope.render_framerate(100)
Asig("samples/snap.wav").stereo().play(rate=0.0825)

In [ ]:
from pya.agen.lib import MouseX, MouseY, SinOsc
SinOsc(600 + MouseX(0,400) * SinOsc(300)).mul(MouseY(0.2, 0)).dup(2).play();

In [ ]:
s.stop()

In [ ]:
s.quit()

In [ ]:
s.scope.exit()